# 📏 长上下文 — 为什么受限，又如何突破 1M

**前置阅读**：建议先读完 `01-theory/02-transformer-architecture.ipynb`（理解 Attention 机制）和 `01-theory/03-transformer-inference.ipynb`（理解 KV Cache）。

**本文目标**：系统理解上下文长度的三大瓶颈，以及当前主流模型突破 1M 上下文的核心技术。

读完这篇你会理解：
- Attention O(n²) 到底有多可怕（128K → 1M 的计算量飞跃）
- KV Cache 为什么是"沉默的显存杀手"
- RoPE 外推为什么失效，PI/NTK/YaRN 如何修复
- Ring Attention、稀疏注意力、KV Cache 量化的分工协作

## 1. 三大瓶颈全景

```
上下文长度从 4K → 128K → 1M，每步跨越都撞上不同的墙:

             4K → 32K             32K → 128K            128K → 1M
             ─────────            ──────────            ───────────
瓶颈 1:
Attention    O(n²) 可忽略         开始感知               严重瓶颈
计算量       0.016B ops/head      1B ops/head            64B ops/head

瓶颈 2:
KV Cache     ~2 MB/请求           ~64 MB/请求            ~500 MB/请求
显存         无压力                需要关注               ← 单请求就 500MB!

瓶颈 3:
位置编码     训练长度内完美         需要 RoPE 扩展         需要全新的位置方案
外推         无问题                (PI/NTK/YaRN)         (或稀疏注意力绕开)
```

### 1.1 瓶颈在不同阶段的表现差异

```
Prefill 阶段 (一次处理全部 prompt):
  瓶颈 1 (计算) → 主要矛盾 ← O(n²) 的 Attention 计算
  瓶颈 2 (显存) → 次要矛盾 (KV Cache 写入是一次性的)
  瓶颈 3 (位置) → 不涉及 (prefill 时位置编码在训练长度内)

Decode 阶段 (逐 token 生成):
  瓶颈 1 (计算) → 次要矛盾 ← Decode 是 O(n), 不是 O(n²)
  瓶颈 2 (显存) → 主要矛盾 ← 每步都要读全部 KV Cache
  瓶颈 3 (位置) → 可能涉及 (如果生成超过训练长度)
```

这解释了为什么不同优化技术针对不同阶段。

## 2. 瓶颈一: Attention 的 O(n²) 计算

### 2.1 具体数字

```python
# 单个 Attention Head 的 FLOPs (简化)
def attention_flops(seq_len, d_head=128):
    # Q @ K^T: [n, d] @ [d, n] = n² × d
    scores = seq_len * seq_len * d_head
    # softmax: ~5 × n² (指数 + 求和 + 除法)
    softmax_ops = 5 * seq_len * seq_len
    # attn @ V: [n, n] @ [n, d] = n² × d
    output = seq_len * seq_len * d_head
    return scores + softmax_ops + output
```

| Context | 每头 FLOPs | 32 头总 FLOPs | A100 理论耗时 |
|---------|-----------|-------------|-------------|
| 4K | 0.016B | 0.5B | ~0.001s |
| 32K | 1B | 32B | ~0.05s |
| 128K | 16B | 512B | ~0.8s |
| 256K | 64B | 2T | ~3.2s |
| 1M | 1T | 32T | **~50s** |

> A100 FP16 理论算力 312 TFLOPS，实际利用率 ~20% for Attention

### 2.2 Prefill 才是重灾区

为什么 Decode 阶段不受 O(n²) 影响？

```
Prefill (处理 prompt，一次性):
  Q: [n, d]   ← 所有 prompt token
  K: [n, d]   ← 所有 prompt token
  Q @ K^T: [n, d] @ [d, n] → [n, n]  ← O(n²)!

Decode (逐 token 生成):
  Q: [1, d]   ← 只有新 token 的 Q
  K: [n, d]   ← 历史全部 KV Cache
  Q @ K^T: [1, d] @ [d, n] → [1, n]  ← O(n), 不是 O(n²)!

所以长上下文推理的 Prefill 延迟是主要矛盾，
而 Decode 的瓶颈在显存带宽 (memory-bound)。
```

### 2.3 解决方案方向

针对 O(n²) 计算，有两条路：

```
路径 A: 减少 Attention 的 n
  → 稀疏注意力: 不是所有 token 之间都做 Attention
  → DeepSeek-V4: SWA + C4(top-512) + C128(dense on 1/128)
  → 实际计算量: O(n × 128 + n × 512 + n × (n/128)) ≈ O(n × 640)

路径 B: 把 n 分到多个 GPU
  → Ring Attention: n 个 GPU 各算 1/n 的序列
  → Striped Attention: 更高效的 GPU 间通信模式
  → 实际计算量: 不变，但并行化了
```

## 3. 瓶颈二: KV Cache — 沉默的显存杀手

### 3.1 为什么长上下文中 KV Cache 比权重还大

```
LLaMA-7B, FP16, MHA (32 KV heads):
  Per token KV Cache = 2 × 32 层 × 32 头 × 128 维 × 2 字节 = 512 KB

Context   KV Cache/请求   10 并发      模型权重
4K        2 GB            20 GB        14 GB  ← KV > 权重
32K       16 GB           160 GB       —     ← OOM
128K      64 GB           640 GB       —     ← 完全不可能
1M        500 GB          5 TB         —     ← 天文数字
```

> 即使 LLaMA-3 8B 使用 GQA (8 KV heads): per-token 128 KB, 1M context = 128 GB/请求

### 3.2 GQA/MQA 能省多少？

```
MHA (32 Q heads, 32 KV heads):
  KV Cache = 32 heads × 2 × layers × dim → 基线

GQA (32 Q heads, 8 KV heads) — LLaMA-2/3:
  4 个 Q head 共享 1 个 KV head
  KV Cache = 8/32 = 25% of MHA → 省 75%

MQA (32 Q heads, 1 KV head) — PaLM:
  所有 Q head 共享 1 个 KV head
  KV Cache = 1/32 = 3% of MHA → 省 97%

但 GQA/MQA 是架构设计时决定的，不能事后加上。
已经在用 MHA 的模型 (如 LLaMA-1)，只能靠其他手段。
```

### 3.3 KV Cache 量化和 Offloading

即使有 GQA，1M context 的 KV Cache 仍然巨大。两条路径：

```
路径 A: KV Cache 量化
  FP16 → INT8: 省 50%
  FP16 → INT4: 省 75%
  
  代价: 轻微精度损失
  llama.cpp 的 Q8_0 KV Cache: 几乎无损
  DeepSeek-V4 的 FP8 KV Cache: 训练时 QAT 补偿

路径 B: KV Cache Offloading
  GPU HBM → CPU DRAM: 容量 x10-100, 但带宽只有 1/40
  GPU HBM → SSD: 容量 x100-1000, 但带宽只有 1/1000
  
  DeepSeek-V4 HiSparse: 只 offload C4 层 (稀疏访问)
  vLLM Swap: offload 完整请求 (应急，非正常态)
```

## 4. 瓶颈三: 位置编码的外推

### 4.1 问题: 训练长度外的位置

```
模型训练时的最大长度 = 4096 (或 8192)
模型在 position 0..4095 见过训练数据
但推理时 position 4096..100000 从未见过!

三种位置编码的外推行为:

Sinusoidal (原始 Transformer):
  理论上可外推 (PE(pos+k) 是 PE(pos) 的线性函数)
  实际: 训练时模型没有学会利用这个性质
  → 直接外推效果差

RoPE (LLaMA/Mistral/Qwen):
  RoPE 在 Attention 计算前对 Q,K 做旋转:
    Q' = rotate(Q, pos × theta)
    K' = rotate(K, pos × theta)
  
  低频分量 (大 theta): 旋转慢, 可以外推
  高频分量 (小 theta): 旋转快, 训练长度外的旋转角度 → 没见过!
  → 直接外推: 高频分量失效, perplexity 飙升

ALiBi (BLOOM):
  Attention 加一个与距离成正比的偏置:
    scores = QK^T - m × |i-j|
  → 天然鼓励"近处更重要"
  → 天然外推性好
  → 但绝对位置信息弱 (不知道自己在序列中的绝对位置)
```

### 4.2 RoPE 扩展方案全景

这是当前最活跃的研究方向，因为几乎所有主流模型都用 RoPE：

```
Position Interpolation (PI, 2023.06):
  思想: 把 position 索引"压缩"回训练长度范围内
    pos' = pos × (L_train / L_target)
  
  如: 训练 4K, 推理 32K → pos' = pos / 8
  缺点: 相邻位置的区分度下降 (位置 80 和 81 → 都映射到 10.0 附近)

NTK-Aware Interpolation (2023.07):
  思想: 只压缩高频分量 (小 theta), 低频分量保持不变
    theta' = theta × scale^(dim/(d/2-1))
  改进: 高频区分度不损失
  缺点: 需要手动调 scale

YaRN (2023.10):
  思想: NTK + 温度调整 + 窗口化
  对极高频分量: 直接截断 (不参与 Attention)
  对不同频段: 分段缩放
  效果: LLaMA-2 7B 从 4K 扩展到 128K, PPL 几乎不变

ReRoPE / Self-Extend (2024):
  思想: 对近处用原始 RoPE (保持局部精度)
       对远处用压缩的 RoPE (保证能算)
  效果: 无需微调, 直接推理时扩展
```

### 4.3 RoPE 扩展对比

| 方法 | 需要微调? | 4K→32K PPL 涨幅 | 4K→128K | 原理 |
|------|----------|-----------------|---------|------|
| 直接外推 | — | +500%+ | 不可用 | 无 |
| PI | 需要 | +5-10% | +20%+ | 全局压缩 |
| NTK | 不需要 | +3-5% | +15% | 仅压缩高频 |
| **YaRN** | 少量(可选) | **+0.5-2%** | **+5-8%** | NTK + 温度 + 窗口 |
| ReRoPE | 不需要 | +2-3% | +10% | 近-远分离 |

> 实际使用：LLaMA-3 官方用 RoPE + 少量长文本继续训练到 128K
> 社区常用 YaRN 或 NTK 把 4K 模型扩展到 32K-128K

## 5. 解决方案全景图

```
                     ┌─────────────────────────────┐
                     │     1M 上下文推理             │
                     └─────────────┬───────────────┘
                                   │
          ┌────────────────────────┼────────────────────────┐
          │                        │                        │
    ┌─────┴─────┐          ┌──────┴──────┐          ┌──────┴──────┐
    │ 计算 O(n²) │          │ 显存 O(n)   │          │ 位置外推    │
    └─────┬─────┘          └──────┬──────┘          └──────┬──────┘
          │                       │                        │
  ┌───────┼──────────┐    ┌───────┼──────────┐    ┌───────┼──────────┐
  │       │          │    │       │          │    │       │          │
  ▼       ▼          ▼    ▼       ▼          ▼    ▼       ▼          ▼
稀疏    Ring       Flash  GQA/  KV Cache   CPU   PI/    YaRN/    RoPE
注意力  Attention  Attn  MQA   量化      Offload NTK    ReRoPE  训练扩展
```

### 5.1 各方案解决的问题

| 方案 | 解决瓶颈 | 收益 | 代价 |
|------|---------|------|------|
| 稀疏注意力 | 计算 O(n²) | 计算量 O(n²)→O(n×K) | 精度可能损失 |
| Ring Attention | 计算 O(n²)+ 显存 | 序列分到多 GPU | 通信开销 + 需要多卡 |
| FlashAttention | 显存(中间量) | 省 90% 激活显存 | 仅省中间结果 |
| GQA/MQA | KV Cache 显存 | 省 75-97% KV | 架构设计时决定 |
| KV Cache 量化 | KV Cache 显存 | 省 50-75% | 轻微精度损失 |
| CPU Offloading | KV Cache 显存 | 容量 x10-100 | 带宽瓶颈 |
| PI/NTK/YaRN | 位置外推 | 4K→32K 几乎无损 | 可能需要微调 |

### 5.2 各模型的方案组合

```
LLaMA-3 (128K):
  ├─ GQA (省 KV Cache 75%)
  ├─ RoPE + 长文本继续训练 (解决外推)
  ├─ FlashAttention (省中间显存)
  └─ 无稀疏注意力 (Full Attention, 计算 O(n²) 仍在)

DeepSeek-V4 (1M):
  ├─ Hybrid Sparse Attention (解决 O(n²))
  │   ├─ SWA (128 tokens, 精确局部)
  │   ├─ C4 (4:1 压缩 + top-512 稀疏全局)
  │   └─ C128 (128:1 压缩 + 全量全局)
  ├─ FP4 专家权重 (省模型显存)
  ├─ HiSparse (C4 层 KV offload 到 CPU)
  ├─ ShadowRadix (混合注意力下的前缀缓存)
  └─ RoPE (原始, 配合训练时扩展)

Gemini 1.5 (1M+):
  ├─ Ring Attention (序列并行)
  ├─ 稀疏 MoE (省计算)
  └─ 内部方案未完全公开

Claude (200K):
  └─ 未公开，推测为稀疏注意力 + RoPE 扩展
```

## 6. Ring Attention — 序列并行的基石

### 6.1 核心思想

```
问题: 1M tokens 的 Attention 一张 GPU 放不下 (KV Cache 太大)

解法: 把序列切成 N 段，N 个 GPU 各负责一段

Ring Attention 的工作流程:

  GPU 0: [token 0..N/K]    GPU 1: [token N/K..2N/K]   ...
  
  Step 1: 每个 GPU 计算自己那段的 Q, K, V
  Step 2: 每个 GPU 把自己的 K, V 发给下一个 GPU (环形)
  Step 3: 收到上一 GPU 的 K, V → 与自己的 Q 做 Attention
  Step 4: 把收到的 K, V 继续往前传
  Step 5: 重复直到 K, V 绕完一圈
  
  结果: 每个 GPU 持有自己段的完整 Attention 输出

关键优化:
  - 计算与通信重叠: GPU 在做 Attention 的同时发送/接收
  - 环形拓扑: 每步只有相邻 GPU 通信，不产生 all-to-all 瓶颈
```

### 6.2 计算量分析

```
无 Ring Attention (单 GPU):
  1M context × 1 GPU → O((1M)²) = 1T ops/head → ~50s

有 Ring Attention (8 GPU):
  每 GPU: (1M/8)² = 15.6B ops/head → ~0.8s
  通信: 8 次 send/recv × (1M/8) KV → ~8 × 64MB
  
  加速比: ~8x (接近线性)
  
  但需要 8 张 GPU → 成本 8x
```

## 7. 代码实验: 上下文长度的代价

In [ ]:
# 上下文长度对推理的影响量化实验

import math

def attention_cost(seq_len, d_head=128, n_heads=32, n_layers=32):
    """计算 Attention 的 FLOPs 和 KV Cache 大小"""
    # Prefill: O(n^2)
    prefill_flops_per_head = seq_len * seq_len * d_head * 2  # QK^T + attn@V
    prefill_total = prefill_flops_per_head * n_heads * n_layers
    
    # Decode: O(n) per step
    decode_flops_per_head = seq_len * d_head * 2
    decode_total = decode_flops_per_head * n_heads * n_layers
    
    # KV Cache (FP16)
    kv_cache_bytes = 2 * n_layers * n_heads * d_head * seq_len * 2  # 2 for K+V, 2 bytes
    
    return prefill_total, decode_total, kv_cache_bytes

print("=" * 70)
print("上下文长度对推理资源的影响")
print("=" * 70)
print(f"{'Length':<10s} {'Prefill FLOPs':<18s} {'Decode/step':<15s} {'KV Cache/req':<15s}")
print("-" * 70)

for n in [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1000000]:
    prefill, decode, kv = attention_cost(n)
    prefill_str = f"{prefill/1e9:.1f}B" if prefill < 1e12 else f"{prefill/1e12:.1f}T"
    kv_str = f"{kv/1e9:.2f} GB" if kv < 1e12 else f"{kv/1e12:.1f} TB"
    print(f"{n:<10,} {prefill_str:<18s} {decode/1e6:>8.1f} M    {kv_str:<15s}")

print()
print("关键拐点:")
print("  32K:  Prefill ~0.5T FLOPs → 在 A100 上 ~1-2s")
print("  128K: Prefill ~8T FLOPs → 需要稀疏注意力或 Ring Attention")
print("  256K: KV Cache ~64GB → 单卡 A100(80GB) 勉强")
print("  1M:   KV Cache ~250GB → 必须多卡或 offloading")

# GQA 的 KV Cache 节省
print()
print("--- GQA 对 KV Cache 的影响 (1M context) ---")
for kv_heads, name in [(32, "MHA"), (8, "GQA (LLaMA-3)"), (4, "GQA x4"), (1, "MQA")]:
    _, _, kv = attention_cost(1000000, n_heads=kv_heads)
    print(f"  {name:20s} ({kv_heads:2d} KV heads): {kv/1e9:.1f} GB")

# KV Cache 量化
print()
print("--- KV Cache 量化效果 (1M context, GQA 8 heads) ---")
base_kv = attention_cost(1000000, n_heads=8)[2]
for dtype, factor, name in [("FP16", 1.0, "FP16"), ("FP8", 0.5, "FP8"), ("INT8", 0.5, "INT8"), ("INT4", 0.25, "Q4_0")]:
    print(f"  {name:10s}: {base_kv*factor/1e9:.0f} GB (节省 {(1-factor)*100:.0f}%)")

# 组合效果
print()
print("--- 组合优化 (1M context) ---")
_, _, kv_gqa_fp8 = attention_cost(1000000, n_heads=8)
kv_opt = kv_gqa_fp8 * 0.5  # GQA 8 heads + FP8
print(f"  GQA(8 heads) + FP8 KV: {kv_opt/1e9:.0f} GB")
print(f"  vs MHA FP16:            节省 {(1 - kv_opt/attention_cost(1000000)[2])*100:.0f}%")
print()
print("结论: GQA + KV Cache 量化是最简单的 '便宜方案'")
print("稀疏注意力是长上下文推理的必然选择")

## 8. 总结

### 三句话记住

1. **计算 O(n²)** → Prefill 的瓶颈 → 稀疏注意力或 Ring Attention
2. **KV Cache O(n)** → Decode 的瓶颈 → GQA + KV 量化 + Offloading
3. **位置外推** → 模型能力的瓶颈 → YaRN/NTK 或 长文本继续训练

### 各层次解决方案的推荐顺序

```
第 0 层 (免费): GQA/MQA
  → 如果选模型时可以控制架构，直接选 GQA 模型

第 1 层 (低成本): KV Cache 量化 (FP16→FP8/INT8)
  → 几乎无损，省 50% 显存，所有框架都支持

第 2 层 (中成本): RoPE 扩展 (YaRN/NTK)
  → 不需要微调，4K→32K 几乎无损

第 3 层 (高成本): 长文本继续训练
  → 4K→128K，需要几千条长文本微调

第 4 层 (架构级): 稀疏注意力 / Ring Attention
  → 1M 级别的必然选择，但需要模型架构支持
```

### 与课程其他章节的关系

- `02-frameworks/vllm/` → PagedAttention 解决了短中长度下的 KV Cache 碎片问题
- `02-frameworks/sglang/` → RadixAttention 通过前缀共享间接减少 KV Cache
- `03-models/deepseek-v4/` → Hybrid Sparse Attention + HiSparse 是 1M 上下文的完整方案
- `01-theory/03-transformer-inference.ipynb` → Prefill/Decode 的计算特性分析